In [1]:
import os
from pathlib import Path
REPO = Path.cwd().resolve().parents[1]          # docs/notebooks -> repo root
GOLDEN = Path(os.environ.get("CAL_DISP_GOLDEN_DIR", REPO / "test_golden"))
SCRATCH = Path(os.environ.get("TMPDIR", REPO / "tmp")) / "cal_disp_notebooks"; SCRATCH.mkdir(parents=True, exist_ok=True)

# Fix 3 — runconfig fields are applied, rejected or warned about

**Defect.** Several PGE runconfig fields were CLI options (`src/cal_disp/cli/config.py`
lines 33–46 and 372–452) and model fields, but nothing read them:
`RunConfig.to_workflow` (`src/cal_disp/config/pge_runconfig.py` lines 204–220) and
`main.run` (`src/cal_disp/main.py` lines 58–107) dropped `mask_file`, `iono_files`,
`tiles_files`, `algorithm_parameters_overrides_json`, `output_format`, `product_path` and
`primary_executable.product_type` without a word, and `_parse_algorithm_overrides`
(`src/cal_disp/config/_utils.py` line 133) was never called. A run configured with an
ionosphere file or frame-specific overrides produced exactly the same product as one
without. Separately, `cal-disp config -c PATH` kept only the file name
(`cli/config.py` line 549) and wrote it into the work directory.

**Fix.**

| field | now |
|---|---|
| `algorithm_parameters_overrides_json` | **applied** in `main.run` (`AlgorithmParameters.with_overrides`); unknown options raise |
| `iono_files`, `tiles_files` | non-empty list **raises** ("not supported in this release") |
| `output_format` | anything but `netcdf` **raises** |
| `mask_file` | **WARNING** in `main.run` (not applied) |
| `product_path` ≠ `sas_output_path`, `product_type` ≠ `DISP_CAL` | **WARNING** in `to_workflow` |
| `compression` | wired in a separate change |
| `cal-disp config -c PATH` | the full path is honoured |

**Demonstrated here.** One script exercises every field with the golden inputs. It is run
twice in a subprocess: against the pre-fix `gamma-release` source (`git archive d0901b3` into scratch) and
against the fixed source. The calibration itself is replaced by a recorder, so the script
shows what `main.run` would hand to `run_calibration`.

In [2]:
import io
import json
import shutil
import subprocess
import sys
import tarfile

import pandas as pd

work = SCRATCH / "10_runconfig_fields"
shutil.rmtree(work, ignore_errors=True)
work.mkdir()

# The pre-fix source tree
old_src = work / "old_src"
tar = subprocess.run(
    ["git", "archive", "--format=tar", "d0901b3", "src/cal_disp"],
    cwd=REPO, check=True, capture_output=True,
).stdout
tarfile.open(fileobj=io.BytesIO(tar)).extractall(old_src, filter="data")
generated = REPO / "src" / "cal_disp" / "_version.py"   # setuptools_scm file, not in git
if generated.exists():
    shutil.copy(generated, old_src / "src" / "cal_disp" / "_version.py")

# Golden inputs (read-only) and the extra files the fields point to
inp = GOLDEN / "input_data"
inputs = {
    "disp_file": str(next((inp / "disp").glob("OPERA_L3_DISP-S1_*.nc"))),
    "unr_grid_latlon_file": str(next((inp / "gnss").glob("grid_latlon_lookup_v*.txt"))),
    "unr_timeseries_dir": str(inp / "gnss"),
    "los_file": str(next((inp / "static_input").glob("*line_of_sight_enu.tif"))),
    "dem_file": str(next((inp / "static_input").glob("*_dem.tif"))),
    "algorithm_parameters_file": str(GOLDEN / "configs" / "algorithm_parameters.yaml"),
    "extra_file": str(work / "extra_input.tif"),
    "overrides_json": str(work / "algorithm_overrides.json"),
}
Path(inputs["extra_file"]).touch()
Path(inputs["overrides_json"]).write_text(
    json.dumps({"8882": {"downsample_factor": 12, "window_size_meters": 300000.0}, "1234": {"downsample_factor": 99}}, indent=2)
)
(work / "inputs.json").write_text(json.dumps(inputs, indent=2))
print(Path(inputs["overrides_json"]).read_text())
print(Path(inputs["algorithm_parameters_file"]).read_text().split("downsample_method")[0])

{
  "8882": {
    "downsample_factor": 12,
    "window_size_meters": 300000.0
  },
  "1234": {
    "downsample_factor": 99
  }
}
calibration_options:
  grid_type: constant
  reference_frame: IGS20
  unwrap_error_correction: false
  apply_tropo_correction: true
  apply_solid_earth_tide_correction: true
  window_size_meters: 600000.0
  posting_meters: 30.0
  downsample_factor: 6
  


## The demo script

Each case sets one field, builds the `RunConfig`, converts it with `to_workflow()` and calls
`main.run` with `run_calibration` replaced by a recorder. The outcome is an error, the
warnings that were logged, or the algorithm parameters that reached the calibration.

In [3]:
DEMO = r'''"""Exercise the runconfig fields; print one JSON line per case."""
import contextlib
import io
import json
import logging
import sys
from pathlib import Path
from unittest.mock import patch

inputs = json.loads(Path(sys.argv[1]).read_text())
out_dir = Path(sys.argv[2])

from click.testing import CliRunner

import cal_disp.main as main
from cal_disp.cli.config import config_cli
from cal_disp.config import DynamicAncillaryFileGroup, InputFileGroup, StaticAncillaryFileGroup
from cal_disp.config.pge_runconfig import OutputOptions, PrimaryExecutable, ProductPathGroup, RunConfig

records = []


class Capture(logging.Handler):
    def emit(self, record):
        if record.levelno >= logging.WARNING:
            records.append(record.getMessage())


logging.getLogger().addHandler(Capture())


def run_case(dynamic=None, static=None, output=None, executable=None, product_path=None):
    """Return what happened for one configuration."""
    records.clear()
    try:
        config = RunConfig(
            input_file_group=InputFileGroup(
                disp_file=inputs["disp_file"], frame_id=8882,
                unr_grid_latlon_file=inputs["unr_grid_latlon_file"],
                unr_timeseries_dir=inputs["unr_timeseries_dir"],
                unr_grid_version="0.3", unr_grid_type="constant",
            ),
            dynamic_ancillary_group=DynamicAncillaryFileGroup(
                algorithm_parameters_file=inputs["algorithm_parameters_file"],
                static_los_file=inputs["los_file"], static_dem_file=inputs["dem_file"],
                **(dynamic or {}),
            ),
            static_ancillary_group=StaticAncillaryFileGroup(**static) if static else None,
            output_options=OutputOptions(**(output or {})),
            primary_executable=PrimaryExecutable(**(executable or {})),
            product_path_group=ProductPathGroup(
                product_path=product_path or out_dir / "output",
                scratch_path=out_dir / "scratch", sas_output_path=out_dir / "output",
            ),
        )
        workflow = config.to_workflow()
        with (
            patch.object(main, "run_calibration") as run_calibration,
            patch.object(main, "make_browse_image_from_nc"),
            contextlib.redirect_stdout(io.StringIO()),
            contextlib.redirect_stderr(io.StringIO()),
        ):
            run_calibration.return_value = out_dir / "output" / "product.nc"
            main.run(workflow)
        cal = run_calibration.call_args.kwargs["algorithm_parameters"].calibration_options
        return {
            "error": None,
            "warnings": list(records),
            "downsample_factor": cal.downsample_factor,
            "window_size_meters": cal.window_size_meters,
        }
    except Exception as exc:  # pydantic ValidationError
        msg = str(exc).splitlines()
        text = " ".join(s.strip() for s in msg[1:3]).split(" [type=")[0]
        return {"error": f"{type(exc).__name__}: {text}", "warnings": list(records)}


extra = inputs["extra_file"]
cases = {
    "baseline (no optional field)": {},
    "iono_files=[file]": {"dynamic": {"iono_files": [extra]}},
    "tiles_files=[file]": {"dynamic": {"tiles_files": [extra]}},
    "output_format='hdf5'": {"output": {"output_format": "hdf5"}},
    "mask_file=file": {"dynamic": {"mask_file": extra}},
    "product_path != sas_output_path": {"product_path": out_dir / "pge_products"},
    "product_type='CUSTOM'": {"executable": {"product_type": "CUSTOM"}},
    "algorithm_parameters_overrides_json": {
        "static": {"algorithm_parameters_overrides_json": inputs["overrides_json"]}
    },
}
for name, kwargs in cases.items():
    print(json.dumps({"case": name, **run_case(**kwargs)}))

# cal-disp config -c PATH: where does the file end up?
requested = out_dir / "configs" / "my_runconfig.yaml"
result = CliRunner().invoke(
    config_cli,
    ["-d", inputs["disp_file"], "-f", "8882", "-ul", inputs["unr_grid_latlon_file"],
     "-ud", inputs["unr_timeseries_dir"], "-uv", "0.3", "-a", inputs["algorithm_parameters_file"],
     "--los-file", inputs["los_file"], "--dem-file", inputs["dem_file"],
     "-o", str(out_dir / "output"), "--work-dir", str(out_dir / "cli_work"), "-c", str(requested)],
)
written = sorted(str(p.relative_to(out_dir)) for p in out_dir.rglob("my_runconfig.yaml"))
print(json.dumps({"case": "cal-disp config -c configs/my_runconfig.yaml --work-dir cli_work",
                  "error": None if result.exit_code == 0 else result.output, "warnings": [],
                  "written": written}))
print(json.dumps({"case": "_source", "error": None, "warnings": [], "written": [main.__file__]}))
'''
demo = work / "demo_fields.py"
demo.write_text(DEMO)


def run_demo(src: Path, tag: str) -> dict:
    out = work / tag
    out.mkdir()
    env = {**os.environ, "PYTHONPATH": str(src)}
    proc = subprocess.run(
        [sys.executable, str(demo), str(work / "inputs.json"), str(out)],
        env=env, capture_output=True, text=True,
    )
    assert proc.returncode == 0, proc.stderr[-3000:]
    rows = [json.loads(line) for line in proc.stdout.splitlines() if line.startswith("{")]
    return {row.pop("case"): row for row in rows}


before = run_demo(old_src / "src", "before")
after = run_demo(REPO / "src", "after")
print("before:", before.pop("_source")["written"][0].replace(str(SCRATCH), "<scratch>"))
print("after :", after.pop("_source")["written"][0].replace(str(REPO), "<repo>"))

before: <scratch>/10_runconfig_fields/old_src/src/cal_disp/main.py
after : <repo>/src/cal_disp/main.py


## Before / after

In [4]:
def describe(row: dict) -> str:
    if row["error"]:
        return "ERROR: " + row["error"]
    if "written" in row:
        return "written to " + ", ".join(row["written"])
    parts = []
    if row["warnings"]:
        parts.append("WARNING: " + " | ".join(row["warnings"]))
    parts.append(f"runs with downsample_factor={row['downsample_factor']}, window_size_meters={row['window_size_meters']:g}")
    return "; ".join(parts)


pd.set_option("display.max_colwidth", None)
table = pd.DataFrame(
    {"before (gamma-release)": {k: describe(v) for k, v in before.items()},
     "after (this fix)": {k: describe(v) for k, v in after.items()}}
)
table.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"})

,before (gamma-release),after (this fix)
baseline (no optional field),"runs with downsample_factor=6, window_size_meters=600000","runs with downsample_factor=6, window_size_meters=600000"
iono_files=[file],"runs with downsample_factor=6, window_size_meters=600000","ERROR: ValidationError: Value error, iono_files is not supported in this release (ionospheric correction is not implemented); got 1 file(s). Remove iono_files from the run configuration."
tiles_files=[file],"runs with downsample_factor=6, window_size_meters=600000","ERROR: ValidationError: Value error, tiles_files is not supported in this release (per-tile calibration is not implemented); got 1 file(s). Remove tiles_files from the run configuration."
output_format='hdf5',"runs with downsample_factor=6, window_size_meters=600000","ERROR: ValidationError: output_format Value error, output_format='hdf5' is not supported in this release; the product is always written as 'netcdf'"
mask_file=file,"runs with downsample_factor=6, window_size_meters=600000","WARNING: mask_file (/u/aurora-r0/govorcin/tmp/cal_disp_notebooks/10_runconfig_fields/extra_input.tif) is not applied in this release: only the masks of the DISP product (recommended_mask, water_mask) and the event databases are used; runs with downsample_factor=6, window_size_meters=600000"
product_path != sas_output_path,"runs with downsample_factor=6, window_size_meters=600000","WARNING: product_path_group.product_path (/u/aurora-r0/govorcin/tmp/cal_disp_notebooks/10_runconfig_fields/after/pge_products) is not used by the SAS: products are written to sas_output_path (/u/aurora-r0/govorcin/tmp/cal_disp_notebooks/10_runconfig_fields/after/output); runs with downsample_factor=6, window_size_meters=600000"
product_type='CUSTOM',"runs with downsample_factor=6, window_size_meters=600000","WARNING: primary_executable.product_type='CUSTOM' is not used: this SAS always writes DISP_CAL products; runs with downsample_factor=6, window_size_meters=600000"
algorithm_parameters_overrides_json,"runs with downsample_factor=6, window_size_meters=600000","runs with downsample_factor=12, window_size_meters=300000"
cal-disp config -c configs/my_runconfig.yaml --work-dir cli_work,written to cli_work/my_runconfig.yaml,written to configs/my_runconfig.yaml


In [5]:
for case in table.index:
    print(case)
    print("   before:", table.loc[case, "before (gamma-release)"].replace(str(SCRATCH), "<scratch>"))
    print("   after :", table.loc[case, "after (this fix)"].replace(str(SCRATCH), "<scratch>"))
    print()

silently_ignored = [
    c for c in table.index
    if c != "baseline (no optional field)" and table.loc[c, "before (gamma-release)"] == table.loc["baseline (no optional field)", "before (gamma-release)"]
]
print(f"{len(silently_ignored)} fields made no difference at all before the fix:", silently_ignored)
assert all(table.loc[c, "after (this fix)"] != table.loc["baseline (no optional field)", "after (this fix)"] for c in silently_ignored)

baseline (no optional field)
   before: runs with downsample_factor=6, window_size_meters=600000
   after : runs with downsample_factor=6, window_size_meters=600000

iono_files=[file]
   before: runs with downsample_factor=6, window_size_meters=600000
   after : ERROR: ValidationError: Value error, iono_files is not supported in this release (ionospheric correction is not implemented); got 1 file(s). Remove iono_files from the run configuration.

tiles_files=[file]
   before: runs with downsample_factor=6, window_size_meters=600000
   after : ERROR: ValidationError: Value error, tiles_files is not supported in this release (per-tile calibration is not implemented); got 1 file(s). Remove tiles_files from the run configuration.

output_format='hdf5'
   before: runs with downsample_factor=6, window_size_meters=600000
   after : ERROR: ValidationError: output_format Value error, output_format='hdf5' is not supported in this release; the product is always written as 'netcdf'

mask_file=file

## The applied override, in detail

`main.run` reads the entry of this frame from the JSON file and applies it to the loaded
`AlgorithmParameters`; the result is what is written to the product metadata
(`/metadata/algorithm_parameters_yaml`). An unknown option is an error, not a no-op.

In [6]:
from cal_disp.config._algorithm import AlgorithmParameters
from cal_disp.config._utils import _parse_algorithm_overrides

params = AlgorithmParameters.from_yaml(inputs["algorithm_parameters_file"])
overrides = _parse_algorithm_overrides(inputs["overrides_json"], 8882)
applied = params.with_overrides(overrides)

print("overrides for frame 8882:", overrides)
print("overrides for frame 9999:", _parse_algorithm_overrides(inputs["overrides_json"], 9999), "\n")
a, b = params.calibration_options.model_dump(), applied.calibration_options.model_dump()
print(f"{'option':32s} {'algorithm_parameters.yaml':>26s} {'with overrides':>16s}")
for key in a:
    if a[key] != b[key]:
        print(f"{key:32s} {a[key]!s:>26s} {b[key]!s:>16s}")

try:
    params.with_overrides({"cal_method": "hanning_fft"})
except Exception as exc:
    print("\nunknown option ->", type(exc).__name__ + ":", str(exc).splitlines()[1].strip(), "-", str(exc).splitlines()[2].strip())

overrides for frame 8882: {'downsample_factor': 12, 'window_size_meters': 300000.0}
overrides for frame 9999: {} 

option                            algorithm_parameters.yaml   with overrides
window_size_meters                                 600000.0         300000.0
downsample_factor                                         6               12
window_size_pixels                                    20000            10000

unknown option -> ValidationError: calibration_options.cal_method - Extra inputs are not permitted [type=extra_forbidden, input_value='hanning_fft', input_type=str]
